# Final Comparison and report
Pull all experiment reports, compare models, perturbations, UQ methods, and XAI impact.

In [1]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.visualization import save_figure
from src.utils.report_generator import generate_report, save_report

config = load_config()
reports_path = config['paths']['results_reports']

DATASET_PREFIXES = ['DIABETES', 'ADULT', 'CIFAR10', 'FASHION_MNIST']
TABULAR_DATASETS = ['DIABETES', 'ADULT']
IMAGE_DATASETS = ['CIFAR10', 'FASHION_MNIST']

def get_dataset(filename):
    # Longest prefix first so FASHION_MNIST is not shadowed
    for prefix in sorted(DATASET_PREFIXES, key=len, reverse=True):
        if filename.startswith(prefix + '_'):
            return prefix
    return None

def load_report(filename):
    with open(os.path.join(reports_path, filename), 'r') as f:
        return json.load(f)

def get_result(report):
    for stage in ['stage1', 'stage2', 'stage3']:
        if stage in report and 'result' in report[stage]:
            return report[stage]['result']
    return None

report_files = [f for f in os.listdir(reports_path)
                if f.endswith('.json') and get_dataset(f) is not None]

found = sorted(set(get_dataset(f) for f in report_files))
print(f"Loaded {len(report_files)} reports across datasets: {found}")

Loaded 60 reports across datasets: ['ADULT', 'CIFAR10', 'DIABETES', 'FASHION_MNIST']


 ### Baselines across all four datasets

In [2]:
BASELINE_TEST_CASES = ['TC_01', 'TC_02', 'TC_03', 'TC_10', 'TC_11']

MODEL_DISPLAY = {
    'Random Forest': 'RF',
    'XGBoost': 'XGB',
    'FCNN': 'FCNN',
    'Simple CNN': 'CNN',
    'ResNet18': 'ResNet',
}

baselines = []
baseline_results = {}

baseline_files = sorted([f for f in report_files
                         if any(f'_{tc}_' in f for tc in BASELINE_TEST_CASES)])

for f in baseline_files:
    r = get_result(load_report(f))
    if r is None:
        continue

    ds = get_dataset(f)
    model_full = r.get('model', 'Unknown')
    model_name = MODEL_DISPLAY.get(model_full, model_full)
    cr = r['classification_report']

    baselines.append({
        'Dataset': ds,
        'Model': model_name,
        'Accuracy': r['accuracy'],
        'Precision': round(cr['macro avg']['precision'], 4),
        'Recall': round(cr['macro avg']['recall'], 4),
        'F1-score': round(cr['macro avg']['f1-score'], 4),
        'Consistency': r.get('consistency', {}).get('mean', 'N/A'),
        'MC_Unc_Mean': r.get('mc_uncertainty', {}).get('mean', 'N/A'),
        'DE_Unc_Mean': r.get('de_uncertainty', {}).get('mean', 'N/A'),
    })
    baseline_results[(ds, model_name)] = r

df_baselines = pd.DataFrame(baselines)
print(f"Found {len(df_baselines)} baselines")
print(df_baselines.to_string(index=False))

Found 10 baselines
      Dataset  Model  Accuracy  Precision  Recall  F1-score Consistency MC_Unc_Mean  DE_Unc_Mean
        ADULT     RF    0.8692     0.8408  0.7802    0.8037       0.606         N/A 0.000000e+00
        ADULT    XGB    0.8773     0.8466  0.8010    0.8201       0.502         N/A 1.000000e-08
        ADULT   FCNN    0.8538     0.8048  0.7796    0.7908       0.617     0.04354 1.626903e-02
      CIFAR10    CNN    0.7607     0.7608  0.7607    0.7594         N/A    0.030484 4.196299e-02
      CIFAR10 ResNet    0.7002     0.7015  0.7002    0.7000         N/A         0.0 5.622838e-02
     DIABETES     RF    0.8644     0.7162  0.5554    0.5664       0.713         N/A 0.000000e+00
     DIABETES    XGB    0.8655     0.7185  0.5721    0.5906        0.65         N/A 1.000000e-08
     DIABETES   FCNN    0.8643     0.7116  0.5666    0.5828       0.593    0.027394 1.232401e-02
FASHION_MNIST    CNN    0.9248     0.9258  0.9248    0.9249         N/A     0.00626 1.210027e-02
FASHION_MNI

### Build every results table

In [3]:
# File groups by test case
mv_files = sorted([f for f in report_files if '_TC_04_' in f])
ci_files = sorted([f for f in report_files if '_TC_05_' in f])
ni_files = sorted([f for f in report_files if '_TC_06_' in f])
all_perturbation_files = mv_files + ci_files + ni_files

tv_files = sorted([f for f in report_files
                   if any(tc in f for tc in ['_TC_07_', '_TC_08_', '_TC_09_'])])

all_tabular_experiment_files = sorted([f for f in report_files
                                       if any(tc in f for tc in
                                              ['_TC_04_', '_TC_05_', '_TC_06_',
                                               '_TC_07_', '_TC_08_', '_TC_09_'])])

img_files = sorted([f for f in report_files
                    if any(tc in f for tc in ['_TC_10_', '_TC_11_', '_TC_12_'])])


# Sheet 2: Perturbations
perturbations = []
for f in all_perturbation_files:
    r = get_result(load_report(f))
    cr = r['classification_report']
    perturbations.append({
        'Dataset': get_dataset(f),
        'Experiment': f.replace('.json', ''),
        'Perturbation': r.get('perturbation', ''),
        'Accuracy': r['accuracy'],
        'Precision': round(cr['macro avg']['precision'], 4),
        'Recall': round(cr['macro avg']['recall'], 4),
        'F1-score': round(cr['macro avg']['f1-score'], 4),
        'MC_Unc_Mean': r.get('mc_uncertainty', {}).get('mean', np.nan),
        'DE_Unc_Mean': r.get('de_uncertainty', {}).get('mean', np.nan),
        'Consistency': r.get('consistency', {}).get('mean', np.nan),
    })
df_perturbations = pd.DataFrame(perturbations)


# Sheet 3: Training variations
training_variations = []
for f in tv_files:
    r = get_result(load_report(f))
    cr = r['classification_report']
    training_variations.append({
        'Dataset': get_dataset(f),
        'Experiment': f.replace('.json', ''),
        'Accuracy': r['accuracy'],
        'Precision': round(cr['macro avg']['precision'], 4),
        'Recall': round(cr['macro avg']['recall'], 4),
        'F1-score': round(cr['macro avg']['f1-score'], 4),
        'MC_Unc_Mean': r.get('mc_uncertainty', {}).get('mean', np.nan),
        'DE_Unc_Mean': r.get('de_uncertainty', {}).get('mean', np.nan),
        'Consistency': r.get('consistency', {}).get('mean', np.nan),
    })
df_training = pd.DataFrame(training_variations)


# Sheet 4: Flagging
flagging_rows = []
for f in sorted(report_files):
    r = get_result(load_report(f))
    if r is None:
        continue
    mc_flag = r.get('flagging_mc', r.get('flagging', {}))
    de_flag = r.get('flagging_de', {})

    flagging_rows.append({
        'Dataset': get_dataset(f),
        'Experiment': f.replace('.json', ''),
        'MC_GREEN_count': mc_flag.get('GREEN', {}).get('count', 0),
        'MC_GREEN_acc': mc_flag.get('GREEN', {}).get('accuracy', 0),
        'MC_YELLOW_count': mc_flag.get('YELLOW', {}).get('count', 0),
        'MC_YELLOW_acc': mc_flag.get('YELLOW', {}).get('accuracy', 0),
        'MC_RED_count': mc_flag.get('RED', {}).get('count', 0),
        'MC_RED_acc': mc_flag.get('RED', {}).get('accuracy', 0),
        'DE_GREEN_count': de_flag.get('GREEN', {}).get('count', 0),
        'DE_GREEN_acc': de_flag.get('GREEN', {}).get('accuracy', 0),
        'DE_YELLOW_count': de_flag.get('YELLOW', {}).get('count', 0),
        'DE_YELLOW_acc': de_flag.get('YELLOW', {}).get('accuracy', 0),
        'DE_RED_count': de_flag.get('RED', {}).get('count', 0),
        'DE_RED_acc': de_flag.get('RED', {}).get('accuracy', 0),
    })
df_flagging = pd.DataFrame(flagging_rows)


# Sheet 5: UQ vs UQ + XAI
xai_rows = []

# FCNN baselines for both tabular datasets
for ds in TABULAR_DATASETS:
    matches = [f for f in report_files if f.startswith(f'{ds}_TC_03_')]
    if len(matches) == 0:
        continue
    r = get_result(load_report(matches[0]))
    xai_rows.append({
        'Dataset': ds,
        'Experiment': matches[0].replace('.json', ''),
        'Type': 'Tabular',
        'MC_XAI_Improvement': r.get('mc_vs_mc_plus_xai', 0),
        'DE_XAI_Improvement': r.get('de_vs_de_plus_xai', 0),
    })

# Tabular experiments
for f in all_tabular_experiment_files:
    r = get_result(load_report(f))
    xai_rows.append({
        'Dataset': get_dataset(f),
        'Experiment': f.replace('.json', ''),
        'Type': 'Tabular',
        'MC_XAI_Improvement': r.get('mc_vs_mc_plus_xai', 0),
        'DE_XAI_Improvement': r.get('de_vs_de_plus_xai', 0),
    })

# Image experiments
for f in img_files:
    r = get_result(load_report(f))
    xai_rows.append({
        'Dataset': get_dataset(f),
        'Experiment': f.replace('.json', ''),
        'Type': 'Image',
        'MC_XAI_Improvement': r.get('mc_vs_mc_plus_xai', 0),
        'DE_XAI_Improvement': r.get('de_vs_de_plus_xai', 0),
    })

df_xai = pd.DataFrame(xai_rows)


# Sheet 6: Image
image_rows = []
for f in img_files:
    r = get_result(load_report(f))
    image_rows.append({
        'Dataset': get_dataset(f),
        'Experiment': f.replace('.json', ''),
        'Accuracy': r['accuracy'],
        'MC_Unc_Mean': r.get('mc_uncertainty', {}).get('mean', np.nan),
        'DE_Unc_Mean': r.get('de_uncertainty', {}).get('mean', np.nan),
        'GradCAM_Correct': r.get('gradcam_feedback', {}).get('correct', np.nan),
        'GradCAM_Incorrect': r.get('gradcam_feedback', {}).get('incorrect', np.nan),
        'GradCAM_Partial': r.get('gradcam_feedback', {}).get('partial', np.nan),
    })
df_image = pd.DataFrame(image_rows)


# Sheet 7: Performance
performance_rows = []
for f in sorted(report_files):
    r = get_result(load_report(f))
    if r is None:
        continue
    for p in r.get('performance', []):
        performance_rows.append({
            'Dataset': get_dataset(f),
            'Experiment': f.replace('.json', ''),
            'Operation': p['operation'],
            'Time_Seconds': p['time_seconds'],
            'Memory_MB': p['memory_usage'],
        })
df_performance = pd.DataFrame(performance_rows)


print(f"Baselines:          {len(df_baselines)} rows")
print(f"Perturbations:      {len(df_perturbations)} rows")
print(f"Training variation: {len(df_training)} rows")
print(f"Flagging:           {len(df_flagging)} rows")
print(f"XAI:                {len(df_xai)} rows")
print(f"Image:              {len(df_image)} rows")
print(f"Performance:        {len(df_performance)} rows")

Baselines:          10 rows
Perturbations:      18 rows
Training variation: 24 rows
Flagging:           60 rows
XAI:                56 rows
Image:              12 rows
Performance:        404 rows


### Export every sheet to one Excel workbook

In [4]:
excel_path = os.path.join(reports_path, 'All_Results_Summary.xlsx')
with pd.ExcelWriter(excel_path) as writer:
    df_baselines.to_excel(writer, sheet_name='Baseline', index=False)
    df_perturbations.to_excel(writer, sheet_name='Perturbations', index=False)
    df_training.to_excel(writer, sheet_name='Training Variations', index=False)
    df_flagging.to_excel(writer, sheet_name='Flagging', index=False)
    df_xai.to_excel(writer, sheet_name='XAI', index=False)
    df_image.to_excel(writer, sheet_name='Image', index=False)
    df_performance.to_excel(writer, sheet_name='Performance', index=False)

print(f"Saved: {excel_path}")

Saved: results/reports\All_Results_Summary.xlsx


### Shorten long experiment names for axis labels

In [5]:
def short_names(series):
    out = series.copy()

    # Strip the dataset prefix
    for prefix in sorted(DATASET_PREFIXES, key=len, reverse=True):
        out = out.str.replace(prefix + '_', '', regex=False)

    # Cosmetic shortening.
    replacements = {
        'Random_Forest_Clean_Baseline': 'RF Baseline',
        'XGBoost_Clean_Baseline': 'XGB Baseline',
        'FCNN_Golden_Baseline': 'FCNN Baseline',
        'Simple_CNN_Golden_Baseline': 'CNN Baseline',
        'ResNet_Golden_Baseline': 'ResNet Baseline',
        'Simple_CNN_': 'CNN ',
        'FCNN_': '',
        'Hyperparameter': 'HP',
        'Class_Imbalance': 'Imbalance',
        'Missing_Values': 'Missing',
        'Noise_Injection': 'Noise',
        'Data_Size': 'Size',
        'Seed_Variation': 'Seed',
        '_': ' ',
    }
    for old, new in replacements.items():
        out = out.str.replace(old, new, regex=False)

    return out.tolist()


def style_axis(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.6)
    ax.set_axisbelow(True)

### Baseline comparison chart, one per dataset

In [15]:
for ds in DATASET_PREFIXES:
    subset = df_baselines[df_baselines['Dataset'] == ds]
    if len(subset) == 0:
        continue

    models = subset['Model'].tolist()
    metrics = {
        'Accuracy': subset['Accuracy'].tolist(),
        'Precision': subset['Precision'].tolist(),
        'Recall': subset['Recall'].tolist(),
        'F1': subset['F1-score'].tolist(),
        'Consistency': pd.to_numeric(subset['Consistency'], errors='coerce').fillna(0).tolist(),
    }

    x = np.arange(len(models))
    width = 0.15
    colors = sns.color_palette('pastel', len(metrics))

    fig, ax = plt.subplots(figsize=(10, 6))
    for i, ((metric_name, values), color) in enumerate(zip(metrics.items(), colors)):
        ax.bar(x + width * i, values, width, color=color, label=metric_name, linewidth=0.5)

    ax.set_ylabel('Score')
    ax.set_title(f'{ds}: Baseline Model Comparison', fontweight='bold', pad=20)
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(models)
    ax.set_ylim(0, 1.1)
    style_axis(ax)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)

    plt.tight_layout()
    save_figure(fig, f'baseline_comparison_{ds.lower()}.png', config)

Saved: results/figures\baseline_comparison_diabetes.png
Saved: results/figures\baseline_comparison_adult.png
Saved: results/figures\baseline_comparison_cifar10.png
Saved: results/figures\baseline_comparison_fashion_mnist.png


### Perturbation uncertainty, one chart per tabular dataset

In [17]:
for ds in TABULAR_DATASETS:
    subset = df_perturbations[df_perturbations['Dataset'] == ds]
    if len(subset) == 0:
        continue

    names = short_names(subset['Experiment'])
    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width / 2, subset['MC_Unc_Mean'], width, color='steelblue', label='MC Dropout')
    ax.bar(x + width / 2, subset['DE_Unc_Mean'], width, color='coral', label='Deep Ensemble')

    base = df_baselines[(df_baselines['Dataset'] == ds) & (df_baselines['Model'] == 'FCNN')]
    if len(base) > 0:
        b_mc = float(base['MC_Unc_Mean'].values[0])
        b_de = float(base['DE_Unc_Mean'].values[0])
        ax.axhline(y=b_mc, color='steelblue', linestyle='--', linewidth=0.8,
                   label=f'MC baseline {b_mc:.4f}')
        ax.axhline(y=b_de, color='coral', linestyle='--', linewidth=0.8,
                   label=f'DE baseline {b_de:.4f}')

    ax.set_ylabel('Mean Uncertainty')
    ax.set_title(f'{ds}: Perturbation Impact on Uncertainty', fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
    top = max(subset['MC_Unc_Mean'].max(), subset['DE_Unc_Mean'].max()) * 1.3
    ax.set_ylim(0, top)
    style_axis(ax)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=False)

    plt.tight_layout()
    save_figure(fig, f'perturbation_uncertainty_{ds.lower()}.png', config)

Saved: results/figures\perturbation_uncertainty_diabetes.png
Saved: results/figures\perturbation_uncertainty_adult.png


### Perturbation consistency, one chart per tabular dataset

In [18]:
for ds in TABULAR_DATASETS:
    subset = df_perturbations[df_perturbations['Dataset'] == ds]
    if len(subset) == 0:
        continue

    names = short_names(subset['Experiment'])
    x = np.arange(len(names))

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x, subset['Consistency'], 0.5, color='indianred', label='Consistency')

    base = df_baselines[(df_baselines['Dataset'] == ds) & (df_baselines['Model'] == 'FCNN')]
    if len(base) > 0:
        b_cons = float(base['Consistency'].values[0])
        ax.axhline(y=b_cons, color='indianred', linestyle='--', linewidth=0.8,
                   label=f'Baseline {b_cons:.4f}')

    ax.set_ylabel('Consistency')
    ax.set_title(f'{ds}: Perturbation Impact on Consistency', fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
    ax.set_ylim(0, 1.0)
    style_axis(ax)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=False)

    plt.tight_layout()
    save_figure(fig, f'perturbation_consistency_{ds.lower()}.png', config)

Saved: results/figures\perturbation_consistency_diabetes.png
Saved: results/figures\perturbation_consistency_adult.png


### Training variation uncertainty, one chart per tabular dataset

In [20]:
for ds in TABULAR_DATASETS:
    subset = df_training[df_training['Dataset'] == ds]
    if len(subset) == 0:
        continue

    names = short_names(subset['Experiment'])
    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.bar(x - width / 2, subset['MC_Unc_Mean'], width, color='steelblue', label='MC Dropout')
    ax.bar(x + width / 2, subset['DE_Unc_Mean'], width, color='coral', label='Deep Ensemble')

    base = df_baselines[(df_baselines['Dataset'] == ds) & (df_baselines['Model'] == 'FCNN')]
    if len(base) > 0:
        b_mc = float(base['MC_Unc_Mean'].values[0])
        b_de = float(base['DE_Unc_Mean'].values[0])
        ax.axhline(y=b_mc, color='steelblue', linestyle='--', linewidth=0.8,
                   label=f'MC baseline {b_mc:.4f}')
        ax.axhline(y=b_de, color='coral', linestyle='--', linewidth=0.8,
                   label=f'DE baseline {b_de:.4f}')

    ax.set_ylabel('Mean Uncertainty')
    ax.set_title(f'{ds}: Training Variation Impact on Uncertainty', fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
    top = max(pd.to_numeric(subset['MC_Unc_Mean'], errors='coerce').max(),
              pd.to_numeric(subset['DE_Unc_Mean'], errors='coerce').max()) * 1.3
    ax.set_ylim(0, top)
    style_axis(ax)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=False)

    plt.tight_layout()
    save_figure(fig, f'training_uncertainty_{ds.lower()}.png', config)

Saved: results/figures\training_uncertainty_diabetes.png
Saved: results/figures\training_uncertainty_adult.png


### XAI impact, one chart per dataset

In [22]:
for ds in DATASET_PREFIXES:
    subset = df_xai[df_xai['Dataset'] == ds]
    if len(subset) == 0:
        continue

    names = short_names(subset['Experiment'])
    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(x - width / 2, subset['MC_XAI_Improvement'], width,
           color='steelblue', label='MC Dropout + XAI')
    ax.bar(x + width / 2, subset['DE_XAI_Improvement'], width,
           color='coral', label='Deep Ensembles + XAI')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

    ax.set_ylabel('GREEN Accuracy Change')
    ax.set_title(f'{ds}: Impact of Adding XAI to UQ', fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
    span = max(abs(subset['MC_XAI_Improvement']).max(),
               abs(subset['DE_XAI_Improvement']).max()) * 1.3
    ax.set_ylim(-span, span)
    style_axis(ax)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=False)

    plt.tight_layout()
    save_figure(fig, f'xai_impact_{ds.lower()}.png', config)

Saved: results/figures\xai_impact_diabetes.png
Saved: results/figures\xai_impact_adult.png
Saved: results/figures\xai_impact_cifar10.png
Saved: results/figures\xai_impact_fashion_mnist.png


### Image uncertainty, one chart per image dataset

In [23]:
for ds in IMAGE_DATASETS:
    subset = df_image[df_image['Dataset'] == ds]
    if len(subset) == 0:
        continue

    names = short_names(subset['Experiment'])
    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(x - width / 2, pd.to_numeric(subset['MC_Unc_Mean'], errors='coerce'),
           width, color='steelblue', label='MC Dropout')
    ax.bar(x + width / 2, pd.to_numeric(subset['DE_Unc_Mean'], errors='coerce'),
           width, color='coral', label='Deep Ensemble')

    base = df_baselines[(df_baselines['Dataset'] == ds) & (df_baselines['Model'] == 'CNN')]
    if len(base) > 0:
        b_mc = float(base['MC_Unc_Mean'].values[0])
        b_de = float(base['DE_Unc_Mean'].values[0])
        ax.axhline(y=b_mc, color='steelblue', linestyle='--', linewidth=0.8,
                   label=f'CNN MC baseline {b_mc:.4f}')
        ax.axhline(y=b_de, color='coral', linestyle='--', linewidth=0.8,
                   label=f'CNN DE baseline {b_de:.4f}')

    ax.set_ylabel('Mean Uncertainty')
    ax.set_title(f'{ds}: Perturbation Impact on Uncertainty', fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
    top = max(pd.to_numeric(subset['MC_Unc_Mean'], errors='coerce').max(),
              pd.to_numeric(subset['DE_Unc_Mean'], errors='coerce').max()) * 1.3
    ax.set_ylim(0, top)
    style_axis(ax)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=False)

    plt.tight_layout()
    save_figure(fig, f'image_uncertainty_{ds.lower()}.png', config)

Saved: results/figures\image_uncertainty_cifar10.png
Saved: results/figures\image_uncertainty_fashion_mnist.png


### Grad-CAM verdict distribution, one chart per image dataset

In [25]:
for ds in IMAGE_DATASETS:
    subset = df_image[df_image['Dataset'] == ds]
    if len(subset) == 0:
        continue

    names = short_names(subset['Experiment'])
    x = np.arange(len(names))
    width = 0.25

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(x - width, pd.to_numeric(subset['GradCAM_Correct'], errors='coerce'),
           width, color='#2d9d78', label='Correct')
    ax.bar(x, pd.to_numeric(subset['GradCAM_Partial'], errors='coerce'),
           width, color='#dfbf00', label='Partial')
    ax.bar(x + width, pd.to_numeric(subset['GradCAM_Incorrect'], errors='coerce'),
           width, color='#e34850', label='Incorrect')

    ax.set_ylabel('Number of reviewed heatmaps')
    ax.set_title(f'{ds}: Grad-CAM Expert Verdicts', fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
    ax.set_ylim(0, 100)
    style_axis(ax)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=3, frameon=False)

    plt.tight_layout()
    save_figure(fig, f'gradcam_verdicts_{ds.lower()}.png', config)

Saved: results/figures\gradcam_verdicts_cifar10.png
Saved: results/figures\gradcam_verdicts_fashion_mnist.png


### Flagging staircase, one chart per dataset

In [26]:
for ds in DATASET_PREFIXES:
    subset = df_flagging[df_flagging['Dataset'] == ds].copy()
    if len(subset) == 0:
        continue

    # keep only rows with a non-empty RED group, so the staircase is visible
    has_all = subset[(subset['MC_RED_count'] > 0) | (subset['DE_RED_count'] > 0)]
    if len(has_all) == 0:
        print(f"{ds}: no experiment has RED flags, skipping staircase chart")
        continue

    # cap at 8 experiments so the chart stays readable
    has_all = has_all.head(8)
    names = short_names(has_all['Experiment'])
    x = np.arange(len(names))
    width = 0.25

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    ax1.bar(x - width, has_all['MC_GREEN_acc'], width, color='#2d9d78', label='GREEN')
    ax1.bar(x, has_all['MC_YELLOW_acc'], width, color='#dfbf00', label='YELLOW')
    ax1.bar(x + width, has_all['MC_RED_acc'], width, color='#e34850', label='RED')
    ax1.set_ylabel('Accuracy')
    ax1.set_title(f'{ds}: MC Dropout Flagging Staircase', fontweight='bold', pad=20)
    ax1.set_xticks(x)
    ax1.set_xticklabels(names, rotation=25, ha='right', fontsize=7)
    ax1.set_ylim(0, 1.2)
    style_axis(ax1)
    ax1.legend(frameon=False)

    ax2.bar(x - width, has_all['DE_GREEN_acc'], width, color='#2d9d78', label='GREEN')
    ax2.bar(x, has_all['DE_YELLOW_acc'], width, color='#dfbf00', label='YELLOW')
    ax2.bar(x + width, has_all['DE_RED_acc'], width, color='#e34850', label='RED')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{ds}: Deep Ensembles Flagging Staircase', fontweight='bold', pad=20)
    ax2.set_xticks(x)
    ax2.set_xticklabels(names, rotation=25, ha='right', fontsize=7)
    ax2.set_ylim(0, 1.2)
    style_axis(ax2)
    ax2.legend(frameon=False)

    plt.tight_layout()
    save_figure(fig, f'flagging_staircase_{ds.lower()}.png', config)

Saved: results/figures\flagging_staircase_diabetes.png
Saved: results/figures\flagging_staircase_adult.png
Saved: results/figures\flagging_staircase_cifar10.png
Saved: results/figures\flagging_staircase_fashion_mnist.png


### Final recommendations, per dataset, and save the summary report

In [27]:
all_recommendations = {}

for ds in DATASET_PREFIXES:
    recs = []

    pert = df_perturbations[df_perturbations['Dataset'] == ds]
    train = df_training[df_training['Dataset'] == ds]
    base_fcnn = df_baselines[(df_baselines['Dataset'] == ds) &
                             (df_baselines['Model'] == 'FCNN')]
    base_cnn = df_baselines[(df_baselines['Dataset'] == ds) &
                            (df_baselines['Model'] == 'CNN')]

    # Model collapse
    for _, row in pd.concat([pert, train]).iterrows():
        if pd.notna(row['F1-score']) and row['F1-score'] < 0.5:
            recs.append(f"Model collapse detected in {row['Experiment']}: "
                        f"macro F1 {row['F1-score']} indicates majority-class-only predictions.")

    # Accuracy vs F1 gap on the clean baseline
    if len(base_fcnn) > 0:
        gap = float(base_fcnn['Accuracy'].values[0]) - float(base_fcnn['F1-score'].values[0])
        if gap > 0.2:
            recs.append(f"Accuracy-F1 gap of {gap:.4f} on the clean baseline. "
                        f"Accuracy alone is misleading for this dataset.")

    # Uncertainty falls while the model collapses
    if len(base_fcnn) > 0:
        base_mc = pd.to_numeric(base_fcnn['MC_Unc_Mean'], errors='coerce').values[0]
        for _, row in pert.iterrows():
            if (pd.notna(row['MC_Unc_Mean']) and pd.notna(row['F1-score'])
                    and row['MC_Unc_Mean'] < base_mc and row['F1-score'] < 0.5):
                recs.append(f"Warning: {row['Experiment']} shows decreased uncertainty "
                            f"despite collapse. Per-prediction UQ is blind to this failure.")

    # Grad-CAM attention quality for image datasets
    if len(base_cnn) > 0:
        cnn_row = df_image[(df_image['Dataset'] == ds) & (df_image['Experiment'].str.contains('TC_10'))]
        if len(cnn_row) > 0:
            correct = pd.to_numeric(cnn_row['GradCAM_Correct'], errors='coerce').values[0]
            incorrect = pd.to_numeric(cnn_row['GradCAM_Incorrect'], errors='coerce').values[0]
            partial = pd.to_numeric(cnn_row['GradCAM_Partial'], errors='coerce').values[0]
            total = correct + incorrect + partial
            if total > 0 and incorrect > total / 2:
                recs.append("CNN Grad-CAM attention was judged incorrect on the majority of "
                            "reviewed samples, suggesting reliance on spurious features.")

    all_recommendations[ds] = recs
    print(f"\n{ds}: {len(recs)} recommendations")
    for i, rec in enumerate(recs):
        print(f"  {i + 1}. {rec}")

final_summary = {'recommendations_by_dataset': all_recommendations}
report_output = generate_report(config, 'Final Comparison Report', stage1_result=final_summary)
save_report(report_output, 'TC_13_Final_Comparison_Report.json', config)
print("\nSaved TC_13_Final_Comparison_Report.json")


DIABETES: 10 recommendations
  1. Model collapse detected in DIABETES_TC_05_FCNN_Class_Imbalance_10pct: macro F1 0.4626 indicates majority-class-only predictions.
  2. Model collapse detected in DIABETES_TC_05_FCNN_Class_Imbalance_30pct: macro F1 0.463 indicates majority-class-only predictions.
  3. Model collapse detected in DIABETES_TC_05_FCNN_Class_Imbalance_50pct: macro F1 0.4889 indicates majority-class-only predictions.
  4. Model collapse detected in DIABETES_TC_06_FCNN_Noise_Injection_50pct: macro F1 0.4626 indicates majority-class-only predictions.
  5. Model collapse detected in DIABETES_TC_08_FCNN_Hyperparameter_LR_0.01_BS_32: macro F1 0.4626 indicates majority-class-only predictions.
  6. Accuracy-F1 gap of 0.2815 on the clean baseline. Accuracy alone is misleading for this dataset.
  7. Warning: DIABETES_TC_05_FCNN_Class_Imbalance_10pct shows decreased uncertainty despite collapse. Per-prediction UQ is blind to this failure.
  8. Warning: DIABETES_TC_05_FCNN_Class_Imbalan